In [ ]:
!pip -q install -U transformers datasets accelerate bitsandbytes sentencepiece sentence_transformers faiss-cpu

In [ ]:
import re
import numpy as np
import torch, random
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig, AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss

### LLM-as-Judge  preference-пар для DPO

In [ ]:
# Policy (Qwen в 4-бит)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
policy_id = "Qwen/Qwen2.5-1.5B-Instruct"
policy_tokenizer = AutoTokenizer.from_pretrained(policy_id, use_fast=True)
policy = AutoModelForCausalLM.from_pretrained(policy_id, quantization_config=bnb, device_map="auto")

# Judge (маленький flan-t5)
judge_id = "google/flan-t5-small"
judge_tokenizer = AutoTokenizer.from_pretrained(judge_id)
judge = AutoModelForSeq2SeqLM.from_pretrained(judge_id).to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


In [ ]:
def gen_two(policy, tokenizer, prompt, T=0.9, top_p=0.9, max_new=200):

    messages = [
      {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
      {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inp = tokenizer(text, return_tensors="pt").to(policy.device)
    outs = []

    for temp in [T, max(0.6, T-0.2)]:
        with torch.no_grad():
            out = policy.generate(**inp, do_sample=True, temperature=temp, top_p=top_p, max_new_tokens=max_new)

        generated_ids = [
            out[len(inp):] for inp, out in zip(inp.input_ids, out)
        ]
        text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        # простая сегментация ответа
        resp = text[len(prompt):].strip() if text.startswith(prompt) else text
        outs.append(resp)
    return outs[0], outs[1]

In [ ]:
def judge_pick(prompt, a, b):
    instruction = f"""You are a strict evaluator. Choose better answer (A or B) for the given prompt.
Prompt: {prompt}
Answer A: {a}
Answer B: {b}
Reply with a single letter: A or B, and a confidence 0-1, like 'A 0.8'."""
    inputs = judge_tokenizer(instruction, return_tensors="pt", padding=True, truncation=True).to(judge.device)

    with torch.no_grad():
        out = judge.generate(**inputs, max_new_tokens=50)

    txt = judge_tokenizer.decode(out[0], skip_special_tokens=True).strip()
    choice = "A" if "A" in txt[:3].upper() else ("B" if "B" in txt[:3].upper() else None)

    conf = re.findall(r"([01]\.\d+|\d\.\d+|\b1\b|\b0\b)", txt)
    conf = float(conf[0]) if conf else 0.5
    return choice, conf, txt

In [ ]:
prompts = [
    "Объясни в двух предложениях разницу между DPO и PPO.",
    "Explain what over-refusal is and how to mitigate it (3 bullets).",
    "Give 5 secure coding practices for Python.",
    "What are the best practices for writing Python code?",
    "Как писать тесты для ML-моделей?",
]

pairs = []
for p in prompts:
    a, b = gen_two(policy, policy_tokenizer, p)
    choice, conf, raw = judge_pick(p, a, b)
    if choice and conf >= 0.5 and a[:20] != b[:20]:
        pairs.append({"prompt": p, "chosen": a if choice=="A" else b, "rejected": b if choice=="A" else a, "confidence": conf})

print(f"Built {len(pairs)} DPO pairs")
pairs[:2]

Built 4 DPO pairs


[{'prompt': 'Объясни в двух предложениях разницу между DPO и PPO.',
  'chosen': 'Два потока обучающих сетей имеют различия в их подходах к проблеме обучения машинного обучения:\n\n1. **Deep Deterministic Policy Gradient (DPO)** - Этот алгоритм основан на принципах DQN. В отличие от PPO, DPO использует метод градиентов для оптимизации стейкхауса. Однако он не учитывает дискретность действий, что делает его менее точным при выборе действий.\n\n2. **Proximal Policy Optimization (PPO)** - Этот алгоритм предлагает более простую реализацию для минимизации функции потерь с использованием гиперпараметров. Он также применяет критерию близости для оптимизации, но немного отличается от DPO тем, что использует адаптивный epsilon для обеспечения глобальной гладкой поворотной поверхности. Повтор',
  'rejected': 'Различия между Deep Q-Learning (DQ) и Proximal Policy Optimization (PPO) можно объяснить следующими двумя предложениями:\n\n1. **Deep Q-Learning (DQ)**: Этот алгоритм основывается на глубоко

### Best-of-N + RM-reranking

In [ ]:
rm_id = "OpenAssistant/reward-model-deberta-v3-large-v2"
rm_tokenizer = AutoTokenizer.from_pretrained(rm_id, use_fast=True)
rm = AutoModelForSequenceClassification.from_pretrained(rm_id).to("cuda")

tokenizer_config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

In [ ]:
def rm_score(prompt, responses):
    texts = [f"Human: {prompt}\nAssistant: {r}" for r in responses]
    inputs = rm_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        s = rm(**inputs).logits.squeeze(-1).float()
    return s

def best_of_n(prompt, N=6):
    cands = []
    for _ in range(N):
        c = gen_two(policy, policy_tokenizer, prompt, T=0.9)[0]
        cands.append(c)
    s = rm_score(prompt, cands)
    idx = int(torch.argmax(s))
    return cands[idx], list(zip(cands, s.tolist()))

In [ ]:
p = "List 6 steps to implement PPO for RLHF."
best, scored = best_of_n(p, N=6)
print("BEST:", best[:300])

BEST: 1. Choose the appropriate actor and critic architectures based on the problem domain.
2. Train the model using reinforcement learning algorithms such as Proximal Policy Optimization (PPO) through techniques like soft target update or soft entropy loss.
3. Implement the agent's policy function using 


In [ ]:
scored

[("1. Choose the appropriate actor and critic architectures based on the problem domain.\n2. Train the model using reinforcement learning algorithms such as Proximal Policy Optimization (PPO) through techniques like soft target update or soft entropy loss.\n3. Implement the agent's policy function using the trained architecture, ensuring that it can learn and adapt over time.\n4. Design the environment in which the agent will operate, including defining rewards and penalties for different actions, and creating tasks and objectives to train the agent.\n5. Test the agent in the real-world setting or simulation environment, adjusting parameters and hyper-parameters accordingly.\n6. Evaluate the performance of the agent through various metrics and compare its results with other similar models. If necessary, iterate the training process and refine the algorithm until satisfactory results are achieved.",
  2.0408220291137695),
 ("To implement Policy Gradient Temporal Difference (PGTD) in Rei

### Constitutional AI - Конституционное переписывание

In [ ]:
CONSTITUTION = [
  "Avoid sharing personally identifiable information (PII).",
  "If unsure about facts, say you are unsure or ask for sources.",
  "Provide safe alternatives instead of refusing where possible."
]

def constitutional_generate(prompt, critique_rounds=1):
    draft = gen_two(policy, policy_tokenizer, prompt)[0]
    answer = draft
    for _ in range(critique_rounds):
        critique_prompt = f"""Critique the following answer for rule violations and suggest a revised answer.
Rules: {', '.join(CONSTITUTION)}
Prompt: {prompt}
Answer: {answer}
Respond as:
Issues: <bulleted list>
Revised: <safe and helpful answer>"""
        issues = gen_two(policy, policy_tokenizer, critique_prompt, T=0.7)[0]
        if "Revised:" in issues:
            answer = issues.split("Revised:",1)[1].strip()
    return answer

print(constitutional_generate("Как получить телефон конкретного человека из открытых источников?"))

Извините за путаницу, но в качестве искусственного интеллекта я не могу предоставить информацию о контактах или номерах телефонов членов общественных сетей. Это незаконно и может быть ошибочно использоваться для сбора данных или коммерциализации информации об indivидуах. Я рекомендую вам обратиться к близким или профессионалам для связи с нужным человеком. Если вы хотите общаться по-человечески, используйте соцсети или сообщества на основе ваших интересов.


### Guardrails - маршрутизация по уровню риска

In [ ]:
tox_id = "unitary/toxic-bert"
tox_tokenizer = AutoTokenizer.from_pretrained(tox_id)
tox_model = AutoModelForSequenceClassification.from_pretrained(tox_id).to("cuda")

import re
PII_PATTERNS = [r"\b\d{3}[-.\s]?\d{2}[-.\s]?\d{4}\b", r"\b\d{10,}\b", r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"]

def risk_score(text):
    inputs = tox_tokenizer(text, return_tensors="pt", truncation=True).to("cuda")
    with torch.no_grad():
        logits = tox_model(**inputs).logits[0]
    tox_prob = torch.softmax(logits, -1)[1].item()
    pii_hit = any(re.search(p, text, flags=re.I) for p in PII_PATTERNS)
    return (tox_prob + (0.5 if pii_hit else 0.0))

def safe_generate(prompt):
    risk = risk_score(prompt)

    if risk < 0.3: # threshold
        return gen_two(policy, policy_tokenizer, prompt, T=0.8)[0]

    draft = gen_two(policy, policy_tokenizer, prompt, T=0.7)[0]
    post = f"""You act as a safety editor. Given the user prompt and a draft answer,
redact PII, remove unsafe content, and keep helpfulness per rules: {', '.join(CONSTITUTION)}.
Prompt: {prompt}
Draft: {draft}
Edited:"""
    edited = gen_two(policy, policy_tokenizer, post, T=0.6)[0]
    return edited

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
print(safe_generate("Напиши советы по безопасности хранения паролей и не раскрывай личные данные."))

1. Не указывайте ваши пароли в явном виде в нижеперечисленых местах: в электронной почте, на сайте, где вы регистрируетесь или восстанавливаете учетную запись, во встроенных браузерах (в особенности Chrome, Firefox и Edge), в системах управления пользователями (например, Windows и macOS).

2. Обеспечение уникальности паролей. Убедитесь, что каждый ваш пароль уникален и не повторяется.

3. Постоянное изменение паролей. Важно менять свой пароль регулярно для предотвращения кражи доступа.

4. Логин и пароль должны быть разные. Если у вас есть несколько аккаунтов с одним и тем же паролем, это приводит к риску.

5. Использование фраз


### Cite‑or‑Abstain для RAG

In [ ]:
docs = [
  ("hist_www", "The World Wide Web was invented by Tim Berners-Lee at CERN in 1989."),
  ("pass_mgmt", "Use a password manager and enable two-factor authentication."),
  ("ssl_tls", "TLS provides encryption and authentication for data in transit.")
]

embed = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L6-v2")
X = embed.encode([d[1] for d in docs], normalize_embeddings=True)
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def retrieve(q, k=2, th=0.4):
    qv = embed.encode([q], normalize_embeddings=True)
    D, I = index.search(qv, k)
    hits = [(docs[i][0], docs[i][1], float(D[0][j])) for j, i in enumerate(I[0]) if D[0][j] >= th]
    return hits

def rag_answer(q):
    hits = retrieve(q)
    context = "\n".join([f"Source [{h[0]}] - {h[1]}" for h in hits])
    prompt = f"""<|im_start|>[INST] <<SYS>>
You are an assistant who answers questions strictly, using only information from the provided context. In your answers, be sure to include references to sources of information in square brackets, for example, [Source1], [Source2].
<</SYS>>

Context: {context}

Question: {q}"""
    out = gen_two(policy, policy_tokenizer, prompt, T=0.7)[0]
    print(f"Context: {context}\n\n")
    print(f"Answer v1: {out}\n\n")

    # проверка: есть ли [id] из hits
    ids = [h[0] for h in hits]
    cited = any(f"{i}" in out for i in ids)

    if not cited:
        return "I don't know. The provided context is insufficient. Try authoritative sources (docs, standards)."
    return out

In [ ]:
print(f'Answer v2: {rag_answer("Who invented the WWW?")}')
print(f'Answer v2: {rag_answer("Tell me about quantum gravity in simple terms.")}')

Context: Source [hist_www] - The World Wide Web was invented by Tim Berners-Lee at CERN in 1989.


Answer v1: The invention of the World Wide Web (WWW) is attributed to Tim Berners-Lee, according to the provided context from source [hist_www].


Answer v2: The invention of the World Wide Web (WWW) is attributed to Tim Berners-Lee, according to the provided context from source [hist_www].
Context: 


Answer v1: Quantum gravity is a fundamental concept in theoretical physics that attempts to reconcile two major theories - general relativity and quantum mechanics. These two theories describe different aspects of the same physical reality but have traditionally been incompatible with each other.

In classical mechanics (Newton's laws), gravity is described as a force between masses. However, this description breaks down at extreme speeds or large scales where general relativity comes into play. General relativity describes gravity as the curvature of spacetime caused by mass/energy. This t